# Step 1: Preprocessing the Taipower Target Data

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the data
df_taipower = pd.read_csv('data/001.csv')

# 1. Rename columns to English immediately
column_mapping = {
    '項次': 'Incident_ID',
    '發生時間': 'Time_Occurred',
    '事故名稱': 'Incident_Description',
    '停電戶數': 'Households_Affected',
    '復電時間': 'Time_Restored'
}
df_taipower.rename(columns=column_mapping, inplace=True)

# 2. Define weather-relevant keywords (Keep these in Chinese to match the raw text)
weather_keywords = ['饋線事故停電', '饋線全停', '分歧開關跳脫', '欠相', '斷線', '過流電驛動作', '差動電驛動作', '51動作', '51N動作', '斷路器跳脫', '越級跳脫']
pattern = '|'.join(weather_keywords)

# 3. Filter for relevant incidents using the new English column name
df_filtered = df_taipower[df_taipower['Incident_Description'].str.contains(pattern, na=False)].copy()

# 4. Parse Timestamps
df_filtered['Timestamp'] = pd.to_datetime(df_filtered['Time_Occurred'], format='%Y%m%dT%H%M%S')

# 5. Create the Binary Target Label
df_filtered['Target_Failure'] = 1 

# 6. Extract Location Entities
import re
def extract_location(text):
    match = re.search(r'(.*?)(變電所|服務區|園區)', text)
    if match:
        return match.group(1) + match.group(2)
    return text

df_filtered['Location_Entity'] = df_filtered['Incident_Description'].apply(extract_location)

# Clean up the dataframe to show only the engineered columns we need
final_columns = ['Incident_ID', 'Timestamp', 'Location_Entity', 'Households_Affected', 'Target_Failure']
df_clean_target = df_filtered[final_columns]

print(df_clean_target.head())

    Incident_ID           Timestamp Location_Entity Households_Affected  \
0             1 2024-12-25 23:27:00           榮成變電所                   1   
4             5 2024-12-10 11:30:00           西螺服務區                  10   
14           15 2024-11-02 10:22:00          中部科學園區                 112   
27           28 2024-10-01 19:30:00          中部科學園區                   1   
28           29 2024-09-28 18:15:00            興變電所               3,230   

    Target_Failure  
0                1  
4                1  
14               1  
27               1  
28               1  


In [3]:
df_clean_target

,Incident_ID,Timestamp,Location_Entity,Households_Affected,Target_Failure
0,1,2024-12-25 23:27:00,榮成變電所,1,1
4,5,2024-12-10 11:30:00,西螺服務區,10,1
14,15,2024-11-02 10:22:00,中部科學園區,112,1
27,28,2024-10-01 19:30:00,中部科學園區,1,1
28,29,2024-09-28 18:15:00,興變電所,"3,230",1
35,36,2024-09-19 01:01:00,中正變電所,"3,093",1
37,38,2024-09-16 17:55:00,成都變電所,"1,915",1
44,45,2024-09-01 05:12:00,新豐配電變電所,1,1
51,52,2024-07-31 02:40:00,內惟配電變電所,253,1
55,56,2024-07-16 14:04:00,成都變電所,"2,350",1


In [4]:
df_filtered

,Incident_ID,Time_Occurred,Incident_Description,Households_Affected,Time_Restored,Timestamp,Target_Failure,Location_Entity
0,1,20241225T232700,榮成變電所第59條饋線全停逾時一小時以上事故,1,20241226T020700,2024-12-25 23:27:00,1,榮成變電所
4,5,20241210T113000,西螺服務區南下休息站欠相停電事故,10,20241210T160000,2024-12-10 11:30:00,1,西螺服務區
14,15,20241102T102200,中部科學園區編號H467饋線事故停電,112,20241102T102200,2024-11-02 10:22:00,1,中部科學園區
27,28,20241001T193000,中部科學園區編號H875饋線事故停電(聯亞科技),1,20241001T194000,2024-10-01 19:30:00,1,中部科學園區
28,29,20240928T181500,興變電所9I28饋線相間過流電驛動作造成饋線斷路器跳脫，事故停電,"3,230",20240928T184200,2024-09-28 18:15:00,1,興變電所
35,36,20240919T010100,中正變電所第51號饋線事故停電,"3,093",20240919T022000,2024-09-19 01:01:00,1,中正變電所
37,38,20240916T175500,成都變電所第62號饋線事故停電,"1,915",20240916T183900,2024-09-16 17:55:00,1,成都變電所
44,45,20240901T051200,新豐配電變電所第59饋線事故停電,1,20240901T051600,2024-09-01 05:12:00,1,新豐配電變電所
51,52,20240731T024000,內惟配電變電所第74饋線斷路器跳脫事故停電,253,20240731T032900,2024-07-31 02:40:00,1,內惟配電變電所
55,56,20240716T140400,成都變電所第62饋線事故停電,"2,350",20240716T140900,2024-07-16 14:04:00,1,成都變電所


In [5]:
locs = df_filtered.Location_Entity.values
locs

array(['榮成變電所', '西螺服務區', '中部科學園區', '中部科學園區', '興變電所', '中正變電所', '成都變電所',
       '新豐配電變電所', '內惟配電變電所', '成都變電所', '永華二次變電所', '關西二次變電所', '源海二次變電所',
       '四湖配電變電所', '新高港超高壓變電所', '九曲配電變電所', '崙背變電所', '常德變電所', '嶺口二次變電所',
       '港南21線事故引起#1M越級跳脫停電', '蘇澳六萬九仟伏特#2匯流排差動電驛動作停電事故', '龍明配電變電所',
       '嶺口二次變電所', '龍明配電變電所', '新工配電變電所'], dtype=object)

# Phase 2: Storm-Centric Sampling & Negative Class Generation

In [11]:
import pandas as pd
import numpy as np
import io

# 1. Load Geomap Data
geomap_csv = """name,type,latitude,longitude,confidence
榮成變電所,substation,25.05482,121.13183,0.95
西螺服務區,place,23.7987,120.4665,0.85
中部科學園區,place,24.2173,120.6997,0.80
興變電所,substation,23.5,120.8,0.40
中正變電所,substation,25.0329,121.5654,0.40
成都變電所,substation,24.9936,121.3010,0.35
新豐配電變電所,substation,24.9053,120.9839,0.75
內惟配電變電所,substation,22.6663,120.2862,0.75
永華二次變電所,substation,22.9908,120.2133,0.75
關西二次變電所,substation,24.7920,121.1760,0.80
源海二次變電所,substation,25.0330,121.5654,0.30
四湖配電變電所,substation,23.6367,120.2250,0.80
新高港超高壓變電所,substation,22.570017,120.407833,0.90
九曲配電變電所,substation,22.9900,120.2000,0.35
崙背變電所,substation,23.7580,120.3530,0.80
常德變電所,substation,25.0600,121.5200,0.30
嶺口二次變電所,substation,22.7580,120.3600,0.75
龍明配電變電所,substation,24.9930,121.2960,0.60
新工配電變電所,substation,24.8138,120.9675,0.60"""

df_geo = pd.read_csv(io.StringIO(geomap_csv)).drop_duplicates(subset=['name'])
df_geo['latitude'] = pd.to_numeric(df_geo['latitude'], errors='coerce')
df_geo['longitude'] = pd.to_numeric(df_geo['longitude'], errors='coerce')
df_geo_clean = df_geo.dropna(subset=['latitude', 'longitude']).copy()

# 2. Load JMA Data & Convert to Local Time (UTC+8)
# Ensure the path matches your actual environment file location
df_storm = pd.read_csv('data/typhoon_data.csv')
df_storm['Timestamp'] = pd.to_datetime(df_storm['Date'], format='%d/%m/%Y %H:%M', errors='coerce')
df_storm['Timestamp'] = df_storm['Timestamp'] + pd.Timedelta(hours=8)
df_storm_2020 = df_storm[df_storm['Timestamp'].dt.year >= 2020].copy()

# 3. Cartesian Join & Distance Math
df_storm_2020['key'], df_geo_clean['key'] = 1, 1
df_cross = pd.merge(df_storm_2020, df_geo_clean, on='key').drop('key', axis=1)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 
    dLat, dLon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dLat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dLon/2)**2
    return R * (2 * np.arctan2(np.sqrt(a), np.sqrt(1-a)))

df_cross['Distance_km'] = haversine(df_cross['latitude'], df_cross['longitude'], df_cross['lat'], df_cross['lng'])
df_danger = df_cross[df_cross['Distance_km'] <= 300.0].copy()

# 4. FIXED Alignment Engine: Using correct Taipower column names
TIME_WINDOW = pd.Timedelta(hours=12)

def verify_failure_fix(geo_name, storm_time, grid_df, window):
    # We use 'Time_Occurred' and 'Incident_Description' as per your df_taipower check
    mask = (
        (grid_df['Time_Occurred'] >= storm_time - window) & 
        (grid_df['Time_Occurred'] <= storm_time + window) & 
        (grid_df['Incident_Description'].str.contains(geo_name, na=False))
    )
    return 1 if mask.any() else 0

print(f"Evaluating {len(df_danger)} localized danger zone events...")

df_taipower['Time_Occurred'] = pd.to_datetime(
    df_taipower['Time_Occurred'], 
    format='%Y%m%dT%H%M%S', 
    errors='coerce'
)

df_taipower['Time_Restored'] = pd.to_datetime(
    df_taipower['Time_Restored'], 
    format='%Y%m%dT%H%M%S', 
    errors='coerce'
)

if df_taipower['Time_Occurred'].isna().any():
    print(f"Warning: {df_taipower['Time_Occurred'].isna().sum()} rows failed to convert.")


# Apply logic using df_taipower which contains 'Time_Occurred' and 'Incident_Description'
df_danger['Target_Failure'] = df_danger.apply(
    lambda row: verify_failure_fix(row['name'], row['Timestamp'], df_taipower, TIME_WINDOW), 
    axis=1
)

print("\n--- PHASE 2 COMPLETE ---")
print(df_danger['Target_Failure'].value_counts())

Evaluating 2819 localized danger zone events...

--- PHASE 2 COMPLETE ---
Target_Failure
0    2819
Name: count, dtype: int64


In [13]:
# Debug Check 1: Do we have 2024 storm data?
print(f"Storm Date Range: {df_storm_2020['Timestamp'].min()} to {df_storm_2020['Timestamp'].max()}")

# Debug Check 2: Do we have 2024 fault data?
print(f"Grid Fault Date Range: {df_taipower['Time_Occurred'].min()} to {df_taipower['Time_Occurred'].max()}")

# Debug Check 3: Can we find even one location manually?
test_name = "榮成變電所"
print(f"Searching for '{test_name}' in Taipower logs...")
matches = df_taipower[df_taipower['Incident_Description'].str.contains(test_name, na=False)]
print(f"Found {len(matches)} matches.")

Storm Date Range: 2020-05-08 14:00:00 to 2022-12-12 20:00:00
Grid Fault Date Range: 2024-01-05 09:14:00 to 2024-12-25 23:27:00
Searching for '榮成變電所' in Taipower logs...
Found 1 matches.


In [14]:
# Quick Alignment Check
storm_years = df_storm['Timestamp'].dt.year.unique()
grid_years = df_taipower['Time_Occurred'].dt.year.unique()

print(f"Storm Years Available: {storm_years}")
print(f"Grid Years Available:  {grid_years}")

common_years = set(storm_years).intersection(set(grid_years))
if not common_years:
    print("❌ ERROR: No overlapping years. You cannot match these datasets.")
else:
    print(f"✅ SUCCESS: Data overlaps in {common_years}. Proceed with Phase 2.")

Storm Years Available: [1978 1979 1981 1982 1983 1984 1985 1986 1987 1988 1989 1990 1991 1992
 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004 2005 2006
 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020
 2021 2022]
Grid Years Available:  [2024]
❌ ERROR: No overlapping years. You cannot match these datasets.


In [15]:
df_taipower.to_csv('data/taipower_data.csv', index=False)

In [ ]:
import pandas as pd
import numpy as np
import io

In [ ]:
# --- 1. Load Geomap Data and Clean ---
geomap_csv = """name,type,latitude,longitude,confidence
榮成變電所,substation,25.05482,121.13183,0.95
西螺服務區,place,23.7987,120.4665,0.85
中部科學園區,place,24.2173,120.6997,0.80
興變電所,substation,23.5,120.8,0.40
中正變電所,substation,25.0329,121.5654,0.40
成都變電所,substation,24.9936,121.3010,0.35
新豐配電變電所,substation,24.9053,120.9839,0.75
內惟配電變電所,substation,22.6663,120.2862,0.75
永華二次變電所,substation,22.9908,120.2133,0.75
關西二次變電所,substation,24.7920,121.1760,0.80
源海二次變電所,substation,25.0330,121.5654,0.30
四湖配電變電所,substation,23.6367,120.2250,0.80
新高港超高壓變電所,substation,22.570017,120.407833,0.90
九曲配電變電所,substation,22.9900,120.2000,0.35
崙背變電所,substation,23.7580,120.3530,0.80
常德變電所,substation,25.0600,121.5200,0.30
嶺口二次變電所,substation,22.7580,120.3600,0.75
港南21線事故引起#1M越級跳脫停電,event,,
蘇澳六萬九仟伏特#2匯流排差動電驛動作停電事故,event,,
龍明配電變電所,substation,24.9930,121.2960,0.60
新工配電變電所,substation,24.8138,120.9675,0.60"""

df_geo = pd.read_csv(io.StringIO(geomap_csv)) 
df_geo = df_geo.drop_duplicates(subset=['name'])

# Force coordinates to numeric (fixes the '0a.75' typo in the raw string)
df_geo['latitude'] = pd.to_numeric(df_geo['latitude'], errors='coerce')
df_geo['longitude'] = pd.to_numeric(df_geo['longitude'], errors='coerce')
df_geo_clean = df_geo.dropna(subset=['latitude', 'longitude']).copy()

In [ ]:
# --- 2. Load and Filter JMA Typhoon Data (2020+) ---
df_storm = pd.read_csv('data/typhoon_data.csv')
df_storm['Timestamp'] = pd.to_datetime(df_storm['Date'], format='%d/%m/%Y %H:%M', errors='coerce')
df_storm = df_storm.dropna(subset=['Timestamp']).copy()
df_storm_2020 = df_storm[df_storm['Timestamp'].dt.year >= 2020].copy()

In [ ]:
# --- 3. Cartesian Product (Cross Join): Match Every Storm Hour to Every Substation ---
# This creates a master matrix where every JMA reading is paired with every infrastructure node
df_storm_2020['key'] = 1
df_geo_clean['key'] = 1
df_cross = pd.merge(df_storm_2020, df_geo_clean, on='key').drop('key', axis=1)

In [ ]:
# --- 4. Calculate Distance & Define Node-Specific Danger Zone ---
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 
    dLat, dLon = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dLat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dLon/2)**2
    return R * (2 * np.arctan2(np.sqrt(a), np.sqrt(1-a)))

# Calculate exact distance from the storm to the specific substation
df_cross['Distance_to_Node_km'] = haversine(
    df_cross['latitude'], df_cross['longitude'], 
    df_cross['lat'], df_cross['lng']
)

# Filter: Only keep rows where the storm was within 300km of THIS specific node
DANGER_RADIUS_KM = 1500.0
df_danger = df_cross[df_cross['Distance_to_Node_km'] <= DANGER_RADIUS_KM].copy()

In [ ]:
# --- 5. The Alignment Engine (Assigning 1s and 0s) ---
TIME_WINDOW = pd.Timedelta(hours=24)

# Note: df_clean_target must be available in memory from Phase 1
def verify_failure(node_name, storm_time, grid_df, window):
    # Filter grid logs for this exact location AND within the time window
    node_faults = grid_df[
        (grid_df['Location_Entity'] == node_name) & 
        (grid_df['Timestamp'] >= storm_time - window) & 
        (grid_df['Timestamp'] <= storm_time + window)
    ]
    return 1 if not node_faults.empty else 0

print(f"Evaluating {len(df_danger)} localized danger zone events...")

# Generate the Target Variable
df_danger['Target_Failure'] = df_danger.apply(
    lambda row: verify_failure(row['name'], row['Timestamp'], df_clean_target, TIME_WINDOW), 
    axis=1
)

In [ ]:
# --- 6. Final QKN Tensor Preparation ---
# Select only the features the QKN will learn from
qkn_features = ['Timestamp', 'name', 'wind', 'pressure', 'Distance_to_Node_km', 'Target_Failure']
df_qkn_final = df_danger[qkn_features].rename(columns={
    'name': 'Node_Name',
    'wind': 'Wind_Speed',
    'pressure': 'Pressure'
})

print("\n--- PHASE 2 COMPLETE ---")
print("Target Class Distribution:")
print(df_qkn_final['Target_Failure'].value_counts())

In [ ]:
df_qkn_final

In [ ]:
df_qkn_final[df_qkn_final['Target_Failure'] == 0]

In [ ]:
df_qkn_final[df_qkn_final['Target_Failure'] == 1]

In [ ]:
set(df_qkn_final['Target_Failure'])